In [1]:
# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
import re
import string
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

from sklearn.utils.class_weight import compute_class_weight

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    GlobalMaxPooling1D,
    SpatialDropout1D
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.utils import to_categorical
# ============================================================
# DOWNLOAD NLTK DATA
# ============================================================

import nltk

resources = [
    'punkt',
    'punkt_tab',
    'stopwords',
    'wordnet',
    'omw-1.4'
]

for resource in resources:
    nltk.download(resource)


# ============================================================
# LOAD DATASET
# ============================================================

# CHANGE PATH ACCORDING TO YOUR FILE
df = pd.read_csv("twitter_training.csv")

# ============================================================
# DATASET CLEANING
# ============================================================

# rename columns if needed
df.columns = ['ID', 'Entity', 'Sentiment', 'Tweet']

# remove missing values
df.dropna(inplace=True)

# remove duplicates
df.drop_duplicates(inplace=True)

print("Dataset Shape:", df.shape)

# ============================================================
# TEXT CLEANING
# ============================================================

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = str(text).lower()

    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove mentions
    text = re.sub(r'@\w+', '', text)

    # remove hashtags
    text = re.sub(r'#', '', text)

    # remove numbers
    text = re.sub(r'\d+', '', text)

    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokenize
    words = word_tokenize(text)

    # remove stopwords + lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]

    return " ".join(words)

# apply cleaning
df['clean_tweet'] = df['Tweet'].apply(clean_text)

print(df[['Tweet', 'clean_tweet']].head())

# ============================================================
# LABEL ENCODING
# ============================================================

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['Sentiment'])

print("\nClasses:")
print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_tweet'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# ============================================================
# TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF Shape:", X_train_tfidf.shape)

# ============================================================
# LOGISTIC REGRESSION MODEL
# ============================================================

lr_model = LogisticRegression(
    max_iter=2000,
    C=4,
    solver='saga'
)

lr_model.fit(X_train_tfidf, y_train)

# predictions
y_pred_lr = lr_model.predict(X_test_tfidf)

# ============================================================
# EVALUATION
# ============================================================

print("\n==============================")
print("LOGISTIC REGRESSION RESULTS")
print("==============================")

print("\nAccuracy:",
      accuracy_score(y_test, y_pred_lr))

print("\nF1 Score:",
      f1_score(y_test, y_pred_lr, average='weighted'))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))

# confusion matrix
cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ============================================================
# DEEP LEARNING SECTION
# ============================================================

# ============================================================
# TOKENIZATION
# ============================================================

MAX_WORDS = 50000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN)

# ============================================================
# ONE HOT ENCODING
# ============================================================

num_classes = len(df['label'].unique())

y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

# ============================================================
# CLASS WEIGHTS
# ============================================================

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print("\nClass Weights:")
print(class_weights)

# ============================================================
# BUILD IMPROVED BiLSTM MODEL
# ============================================================

model = Sequential()

model.add(
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=128,
        input_length=MAX_LEN
    )
)

model.add(SpatialDropout1D(0.3))

model.add(
    Bidirectional(
        LSTM(
            128,
            return_sequences=True
        )
    )
)

model.add(GlobalMaxPooling1D())

model.add(Dense(128, activation='relu'))

model.add(Dropout(0.4))

model.add(Dense(num_classes, activation='softmax'))

# ============================================================
# COMPILE MODEL
# ============================================================

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# ============================================================
# EARLY STOPPING
# ============================================================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# ============================================================
# TRAIN MODEL
# ============================================================

history = model.fit(
    X_train_pad,
    y_train_cat,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weights
)

# ============================================================
# EVALUATE MODEL
# ============================================================

loss, accuracy = model.evaluate(X_test_pad, y_test_cat)

print("\n==============================")
print("BiLSTM RESULTS")
print("==============================")

print("\nTest Accuracy:", accuracy)

# predictions
y_pred_dl = model.predict(X_test_pad)

y_pred_dl = np.argmax(y_pred_dl, axis=1)

print("\nF1 Score:",
      f1_score(y_test, y_pred_dl, average='weighted'))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_dl))

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm_dl = confusion_matrix(y_test, y_pred_dl)

plt.figure(figsize=(8,6))
sns.heatmap(cm_dl, annot=True, fmt='d', cmap='Greens')

plt.title("Confusion Matrix - BiLSTM")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# ============================================================
# TRAINING CURVES
# ============================================================

plt.figure(figsize=(10,5))

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

# ============================================================
# SAVE MODELS
# ============================================================

import joblib

# save tfidf model
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

# save logistic regression
joblib.dump(lr_model, 'logistic_regression_model.pkl')

# save deep learning model
model.save("bilstm_sentiment_model.keras")
print("\nModels Saved Successfully!")

# ============================================================
# INFERENCE FUNCTION
# ============================================================

def predict_sentiment(text):

    cleaned = clean_text(text)

    # TF-IDF prediction
    vector = vectorizer.transform([cleaned])

    pred_lr = lr_model.predict(vector)[0]

    # Deep Learning prediction
    seq = tokenizer.texts_to_sequences([cleaned])

    pad = pad_sequences(seq, maxlen=MAX_LEN)

    pred_dl = model.predict(pad)

    pred_dl = np.argmax(pred_dl, axis=1)[0]

    print("\n==============================")
    print("INPUT:", text)
    print("==============================")

    print("\nLogistic Regression Prediction:",
          encoder.inverse_transform([pred_lr])[0])

    print("BiLSTM Prediction:",
          encoder.inverse_transform([pred_dl])[0])

# ============================================================
# TEST PREDICTIONS
# ============================================================

predict_sentiment("This game is absolutely amazing!")

predict_sentiment("I hate this update. Worst experience ever.")

predict_sentiment("The gameplay is decent but graphics are average.")

ModuleNotFoundError: No module named 'tensorflow'